In [ ]:
import rp
import csv
import time
import numpy as np
from pathlib import Path

print("Imports done")

In [ ]:
#setting real sampling rate from the desired one

sampling_rate_wanted = 32000

clock = 125 * (10**6)

decimation = round(clock / sampling_rate_wanted)

sampling_rate_real = clock / decimation

#setting samples and duration

samples = 32000
sample_duration = samples / sampling_rate_real

output = Path("32000samples.csv") #name of file to make

#the red pitaya returns either 0 or rp_ok on every command sent, so this just makes sure all
#hardware right before collecting data

def check(result, operation):
    if result != rp.RP_OK or 0:
        raise RuntimeError(operation, result)

print("Parameters set")
print("Sampling rate (Hz):", sampling_rate_real)
print("Sample duration (s):", sample_duration)

In [ ]:
#data acquisition

check(rp.rp_Init(), "rp_Init")
print("Connected")
check(rp.rp_AcqReset(), "rp_AcqReset")

#channels are set to +-1/+-20 volts via a physical piece, I have channel 1 set to 20 and 2 set to 1 (untouched from how I got it)

check(rp.rp_AcqSetGain(rp.RP_CH_1, rp.RP_HIGH), "rp_AcqSetGain")

#32k samples is over the 16k buffer, so deep memory is needed

get_memory = rp.rp_AcqAxiGetMemoryRegion()
check(get_memory[0], "rp_AcqAxiGetMemoryRegion")

memory_start = get_memory[1]

check(rp.rp_AcqAxiSetDecimationFactor(decimation), "rp_AcqAxiSetDecimationFactor") #how often to save sample
check(rp.rp_AcqAxiSetTriggerDelay(rp.RP_CH_1, samples), "rp_AcqAxiSetTriggerDelay") #how many samples after trigger
check(rp.rp_AcqAxiSetBufferSamples(rp.RP_CH_1, memory_start, samples),"rp_AcqAxiSetBufferSamples") #where in memory to save them
check(rp.rp_AcqAxiEnable(rp.RP_CH_1, True), "rp_AcqAxiEnable") #allow deep memory on channel 1

#from here starts data acquisition

check(rp.rp_AcqStart(), "rp_AcqStart")
check(rp.rp_AcqSetTriggerSrc(rp.RP_TRIG_SRC_NOW), "rp_AcqSetTriggerSrc")
print("Saving channel 1")


while True:
    #check for if the buffer is full
    buffer_fill = rp.rp_AcqAxiGetBufferFillState(rp.RP_CH_1)
    check(buffer_fill[0], "rp_AcqAxiGetBufferFillState")
    if buffer_fill[1]:
        break

check(rp.rp_AcqStop(), "rp_AcqStop")

memory_end = rp.rp_AcqAxiGetWritePointerAtTrig(rp.RP_CH_1) #since deep memory loops, this finds where it stopped
check(memory_end[0], "rp_AcqAxiGetWritePointerAtTrig")
voltage_data = rp.fBuffer(samples) #make space for data
data_result = rp.rp_AcqAxiGetDataV(rp.RP_CH_1, memory_end[1], samples, voltage_data) #move memory data into the voltage list 
check(data_result[0], "rp_AcqAxiGetDataV")

#write file for data

with output.open("w", newline = "") as file:
    writer = csv.writer(file)
    writer.writerow(["Voltage (V)"])
    for i in range(samples):
        writer.writerow([voltage_data[i]])

print("Run saved")

rp.rp_AcqAxiEnable(rp.RP_CH_1, False)
rp.rp_Release()